In [33]:

import pandas as pd

file_path = "../data/processed/market_data_clean_v2.parquet"

df = pd.read_parquet(file_path)
df.columns


Index(['product_name', 'category', 'bs_year', 'bs_month', 'month_idx',
       'month_name', 'volume', 'min_price', 'max_price', 'avg_price', 'unit',
       'unit_canonical', 'unit_changed', 'total_amount', 'volume_equals',
       'total_sources', 'reconciliation_gap', 'import_share',
       'n_months_present', 'is_balanced', 'baglung', 'bara', 'beni', 'bhutan',
       'birgunj', 'china', 'dang', 'dhading', 'gorkha', 'hetauda', 'india',
       'jhapa', 'kathmandu', 'kevray', 'lahan', 'lamjung', 'local', 'marpha',
       'mugu', 'mustang', 'myagdi', 'naew', 'narayangadh', 'nawalparasi',
       'nuwakot', 'palpa', 'parbat', 'rolpa', 'rukum', 'rupandehi', 'salyan',
       'sarlahi', 'sindhuli', 'sunsari', 'syangja', 'tanahu', 'india_share',
       'china_share', 'bhutan_share', 'domestic_share', 'n_sources',
       'herfindahl', 'm_sin', 'm_cos', 'avg_price_lag1', 'volume_lag1'],
      dtype='object')

In [2]:
import matplotlib.pyplot as plt

# Histogram of Avg_Price
plt.figure(figsize=(8,6))
df['avg_price'].hist(bins=50, edgecolor='white')

# Label axes and title
plt.xlabel("Average Price (per unit)")
plt.ylabel("Number of Observations")
plt.title("Distribution of Average Prices Across Products")

# Save to reports folder
plt.savefig("../reports/avg_price_histogram.png", dpi=300, bbox_inches='tight')
plt.close()


In [5]:
# Select rows where avg_price > 500
mask = df['avg_price'] > 480

# Get unique product names from those rows
unique_products = df.loc[mask, 'product_name'].unique().tolist()

print("Unique products with price > 480:", unique_products)



Unique products with price > 480: ['Akabary_Chilly', 'Avocado', 'Coriander_Green', 'Grapes', 'Kiwi', 'Strawberry']


In [ ]:
import os
print(os.path.exists('../models/price_surrogate_v1.txt'))
print(os.path.exists('../models/price_surrogate_final.txt'))


True
True


## Testing the price model finaal

In [35]:
import lightgbm as lgb
import numpy as np
from features import load_clean_data, get_src_cols, add_lags, get_feature_cols, cat_cols

booster = lgb.Booster(model_file='../models/price_surrogate_final.txt')

d = load_clean_data()
src_cols = get_src_cols(d)   # still needed, just to know which raw columns are the "sources" for feature_cols
feature_cols = get_feature_cols(src_cols)

sample = d[d['product_name'].isin(['Apple', 'Sugarcane', 'Orange_Sweet', 'Potato'])].copy()
sample = sample.sort_values(['product_name', 'month_idx'])

X_sample = sample[feature_cols].copy()
for c in cat_cols:
    X_sample[c] = X_sample[c].astype('category')

pred_log = booster.predict(X_sample)
sample['predicted_price'] = np.expm1(pred_log)
print(sample[['product_name', 'month_idx', 'avg_price', 'predicted_price']])

     product_name  month_idx  avg_price  predicted_price
14          Apple          1     192.40       177.553018
15          Apple          2     289.86       194.591844
16          Apple          3     237.81       282.856791
17          Apple          4     255.00       256.267504
18          Apple          5     255.00       256.713348
19          Apple          6     255.00       254.520338
20          Apple          7     255.00       256.199247
21          Apple          8     255.00       256.043724
22          Apple          9     255.00       255.149773
23          Apple         10     266.30       254.204475
500  Orange_Sweet          1     110.00       142.332426
501  Orange_Sweet          2     110.00       117.788837
502  Orange_Sweet          3     110.00       123.578597
503  Orange_Sweet          4     110.00       104.466936
504  Orange_Sweet          5     110.00       108.142581
505  Orange_Sweet          6     110.00       110.145550
506  Orange_Sweet          7   